# NB 1.1 &mdash; Solucions dels exercicis

**MP 5134** &mdash; UT1 · *Versió amb els pingüins de l'arxipèlag Palmer*

---

Solucionari dels sis exercicis de la secció 7 del
[NB 1.1](NB_1_1_presa_de_contacte_PINGUINS.ipynb).

Cada exercici porta l'enunciat, una solució comentada i, quan hi ha alguna cosa a
dir sobre la correcció, una nota del que val la pena buscar a les respostes de
l'alumnat.

Els exercicis 3, 4 i 5 tenen una única resposta correcta. Els exercicis 2 i 6
demanen una interpretació: allà el que s'avalua és el raonament, no el número.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

URL_DADES = "https://raw.githubusercontent.com/pprohenspolitecnicllevant/disseny-avaluacio-models-ml/refs/heads/main/UT01-Entorn_de_treball_primer_model/penguins/penguins.csv"
df = pd.read_csv(URL_DADES)

print(f"{df.shape[0]} files, {df.shape[1]} columnes")

## Exercici 1

> Quants pingüins es van mesurar cada any? I a cada illa? Fes servir
> `value_counts()` sobre les columnes `year` i `island`.

In [ ]:
# value_counts() ordena per freqüència. Amb els anys interessa més l'ordre
# cronològic, i això ho dóna sort_index(), que ordena per l'etiqueta.
print("Pingüins mesurats cada any:")
print(df["year"].value_counts().sort_index())

print()
print("Pingüins mesurats a cada illa:")
print(df["island"].value_counts())

**Resposta.** Per anys: 110 el 2007, 114 el 2008 i 120 el 2009. Les tres
campanyes van ser de mida molt semblant, cosa que és bon senyal: si un any
n'hi hagués deu i un altre tres-cents, hauríem de preguntar-nos què va passar
abans de barrejar-los.

Per illes el repartiment és molt més desigual: **Biscoe 168, Dream 124 i
Torgersen 52**. A Torgersen s'hi van mesurar una quarta part dels pingüins que a
Biscoe.

*Per a la correcció:* qui faci servir `value_counts()` sense `sort_index()` a la
columna dels anys obtindrà 2009, 2008, 2007. No és un error, però val la pena
fer notar la diferència entre ordenar per freqüència i ordenar per etiqueta.

## Exercici 2

> Totes les espècies viuen a totes les illes? Pista:
> `pd.crosstab(df["species"], df["island"])`. Si la resposta et sorprèn, pensa
> què implicaria per a un model que hagués de predir l'espècie sabent l'illa.

In [ ]:
# crosstab() creua dues columnes categòriques i compta quantes files hi ha a
# cada combinació. És la taula de contingència de tota la vida.
taula = pd.crosstab(df["species"], df["island"])
taula

**Resposta: no, i de molt lluny.**

| | Biscoe | Dream | Torgersen |
|---|---|---|---|
| **Adelie** | 44 | 56 | 52 |
| **Chinstrap** | 0 | 68 | 0 |
| **Gentoo** | 124 | 0 | 0 |

Els **Gentoo només viuen a Biscoe**. Els **Chinstrap només a Dream**. Els Adelie
són els únics que apareixen a les tres illes.

Ara la part important, que és la segona meitat de l'enunciat.

In [ ]:
# Quant encertaria un model que NOMÉS mirés l'illa i digués sempre l'espècie
# més freqüent d'aquella illa? idxmax() dóna l'etiqueta de la fila més gran.
for illa in taula.columns:
    especie = taula[illa].idxmax()
    encerts = taula.loc[especie, illa]
    total = taula[illa].sum()
    print(f"{illa:<10} -> diria '{especie}': encerta {encerts} de {total}")

print()
total_encerts = sum(taula[illa].max() for illa in taula.columns)
print(f"Encerts totals: {total_encerts} de {taula.values.sum()} "
      f"({100 * total_encerts / taula.values.sum():.1f} %)")

Un model que no mira **cap** mesura del pingüí, que només pregunta a quina illa
és, encerta el **70,9 %** de les vegades. I a Torgersen encerta el 100 %, perquè
allà tots els pingüins són Adelie.

Aquí hi ha la lliçó, i és de les que valen per a tot el curs: **una variable pot
semblar molt informativa i el que estar fent en realitat és delatar la resposta**.
L'illa no explica *per què* un pingüí és d'una espècie o d'una altra; simplement
codifica on viu cadascuna. Un model entrenat amb l'illa aprendria geografia, no
biologia, i el dia que li presentessin un pingüí d'una illa nova no sabria què
fer-ne.

No és exactament una fuga d'informació &mdash; l'illa és un dada legítima que es
coneix abans de saber l'espècie &mdash; però és el mateix reflex el que ens ha de
saltar: quan una variable sola dóna un resultat sospitosament bo, cal preguntar-se
què està capturant de veritat.

*Per a la correcció:* la resposta completa ha de dir les dues coses, que les
espècies no es reparteixen per igual **i** que això faria el problema massa fàcil
d'una manera poc útil. Qui només copiï la taula ha fet la meitat de l'exercici.

## Exercici 3

> Quin va ser el pingüí més pesant de tota la sèrie? I el del bec més llarg?
> Pista: `idxmax()` et dóna la posició del màxim, i `df.loc[...]` et dóna aquella
> fila.

In [ ]:
# idxmax() no dóna el valor màxim (això és max()), sinó l'ÍNDEX de la fila on
# es troba. Amb aquest índex, .loc[] ens retorna la fila sencera.
idx_pesant = df["body_mass_g"].idxmax()
print("El pingüí més pesant és la fila", idx_pesant)
print(df.loc[idx_pesant])

In [ ]:
idx_bec = df["bill_length_mm"].idxmax()
print("El del bec més llarg és la fila", idx_bec)
print(df.loc[idx_bec])

**Resposta.** Els dos rècords són **Gentoo mascles de l'illa Biscoe, mesurats el
2007**:

- **El més pesant** és la fila 169: 6.300 g, amb una aleta de 221 mm.
- **El del bec més llarg** és la fila 185: 59,6 mm de bec i 6.050 g.

Que tots dos siguin Gentoo mascles no és casualitat, i connecta amb l'exercici 4:
els Gentoo són l'espècie gran, i dins de cada espècie els mascles són més grossos
que les femelles.

*Per a la correcció:* l'error típic és fer servir `max()` en comptes de
`idxmax()` i quedar-se amb el número 6300 sense saber de quin pingüí és. Val la
pena insistir-hi, perquè la diferència entre *el valor* i *on és el valor*
reapareix constantment.

## Exercici 4

> Calcula la massa mitjana de cada espècie i dibuixa-la amb un gràfic de barres.
> Pista: `df.groupby("species")["body_mass_g"].mean()`.

In [ ]:
# groupby() parteix la taula en grups segons el valor d'una columna i després
# aplica una operació a cada grup. Aquí: agrupa per espècie i fes la mitjana
# de la massa. Els valors absents els ignora tot sol.
massa_mitjana = df.groupby("species")["body_mass_g"].mean()
print(massa_mitjana.round(0))

In [ ]:
# El resultat d'un groupby és una Series, i les Series de Pandas es dibuixen
# soles amb .plot(). kind="bar" demana un gràfic de barres.
massa_mitjana.plot(kind="bar", figsize=(7, 4), edgecolor="white", rot=0)

plt.ylabel("Massa mitjana (g)")
plt.xlabel("")
plt.title("Massa corporal mitjana per espècie")
plt.show()

**Resposta.**

| Espècie | Massa mitjana |
|---|---|
| Adelie | 3.701 g |
| Chinstrap | 3.733 g |
| Gentoo | **5.076 g** |

Adelie i Chinstrap pesen pràcticament el mateix &mdash; 32 grams de diferència
sobre 3.700, menys d'un 1 % &mdash; i els Gentoo pesen un terç més que
qualsevol dels dos.

És la confirmació numèrica del que es veia a l'histograma de la secció 6: els dos
cims no eren tres espècies, eren **els Gentoo contra tots els altres**.

*Per a la correcció:* alerta amb qui dibuixi el gràfic amb l'eix vertical
començant a 3.500 (matplotlib no ho fa per defecte, però algunes eines sí). La
diferència sembla llavors abismal. Bon moment per comentar que un eix retallat
exagera qualsevol diferència.

## Exercici 5

> Repeteix el mapa de dispersió de la secció 6 canviant els eixos per
> `flipper_length_mm` i `body_mass_g`. Separa igual de bé les espècies?

In [ ]:
plt.figure(figsize=(7, 6))

for especie in df["species"].unique():
    subconjunt = df[df["species"] == especie]
    plt.scatter(subconjunt["flipper_length_mm"], subconjunt["body_mass_g"],
                s=14, alpha=0.7, label=especie)

plt.xlabel("Llargada de l'aleta (mm)")
plt.ylabel("Massa corporal (g)")
plt.title("L'aleta i la massa separen les espècies?")
plt.legend()
plt.show()

In [ ]:
# Els rangs de cada espècie, per posar números a el que es veu al dibuix.
df.groupby("species")[["flipper_length_mm", "body_mass_g"]].agg(["min", "max"])

**Resposta: separa, però només a mitges.**

Els **Gentoo queden perfectament aïllats**: les seves aletes van de 203 a 231 mm i
les de les altres dues espècies no passen de 212 mm. Amb aquesta única variable ja
els identificaries.

**Adelie i Chinstrap, en canvi, se superposen del tot.** Mira els rangs:

| | Aleta | Massa |
|---|---|---|
| Adelie | 172 &ndash; 210 mm | 2.850 &ndash; 4.775 g |
| Chinstrap | 178 &ndash; 212 mm | 2.700 &ndash; 4.800 g |

Són pràcticament idèntics. Cap frontera traçada en aquest dibuix els podria
distingir.

Comparat amb el mapa dels becs de la secció 6, que separava **les tres** espècies,
aquest és clarament pitjor. I la conclusió és la que ens interessa: **no totes les
variables serveixen igual per a un problema**. La massa i l'aleta mesuren la mida
general de l'animal, i Adelie i Chinstrap són igual de grossos; el que els
diferencia és la *forma* del bec, no la talla.

Escollir quines variables entren al model és una decisió amb conseqüències, i té
tota una unitat dedicada: la **UT3**.

*Per a la correcció:* la resposta ha de distingir els dos casos. Un simple "sí" o
"no" no val: els Gentoo sí, els altres dos no.

## Exercici 6

> Pensa i escriu la resposta: si volguéssim predir el **sexe** d'un pingüí, quines
> columnes creus que serien més útils? I creus que seria un problema més fàcil o
> més difícil que endevinar l'espècie?

Aquest exercici era de reflexió, sense codi. Però com que la resposta es pot
comprovar, val la pena tenir els números a mà per al debat.

In [ ]:
# Dins de cada espècie, quant pesen els mascles i les femelles?
df.groupby(["species", "sex"])["body_mass_g"].mean().round(0)

**Quines columnes.** Totes les mesures del cos serveixen, perquè **dins de cada
espècie els mascles són clarament més grossos**: 674 g més els Adelie, 412 g més
els Chinstrap i 805 g més els Gentoo.

Però fixa't en el detall que fa interessant el problema: un pingüí de 4.000 g pot
ser un **Adelie mascle** (mitjana 4.043 g) o un **Gentoo femella** (mitjana
4.680 g, però amb exemplars molt més lleugers). **La massa sola no decideix res si
no saps l'espècie.** Per tant, la columna més útil de totes no és cap mesura: és
`species`.

**Més fàcil o més difícil?** Més difícil, per dos motius.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.dummy import DummyClassifier

MESURES = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]
dades = df[MESURES + ["sex"]].dropna()

X_train, X_test, y_train, y_test = train_test_split(
    dades[MESURES], dades["sex"], test_size=0.2, random_state=42
)

arbre = DecisionTreeClassifier(max_depth=4, random_state=42).fit(X_train, y_train)
ref = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)

print("Predir el SEXE amb les quatre mesures")
print(f"  model de referència: {ref.score(X_test, y_test):.3f}")
print(f"  arbre de decisió:    {arbre.score(X_test, y_test):.3f}")

**Els números.** L'arbre encerta el **86,6 %** del sexe, contra el **94,2 %** que
treia amb l'espècie al NB 1.2. Efectivament, més difícil.

I hi ha una segona diferència, més subtil i més important, que es veu comparant
els models de referència. Amb l'espècie, la referència ja partia del 50 % perquè
els Adelie són la meitat de les mostres. Amb el sexe, mascles i femelles estan
repartits al 50,5 % contra 49,5 %: **el problema està perfectament equilibrat**.

El model de referència, de fet, treu **0,463** sobre aquest conjunt de prova: no
és exactament 0,5 perquè en aquestes 67 files concretes hi han caigut una mica
més de femelles que de mascles, però és el que s'espera de tirar una moneda.

Això vol dir que aquí el 86,6 % és un 86,6 % honest: el model ha hagut de guanyar-
se cada encert. Quan a la **UT4** discutim per què el percentatge d'encerts enganya
amb classes desbalancejades, aquest cas servirà de contraexemple: és el problema on
sí que se'n pot fiar un.

*Per a la correcció:* la millor resposta possible és la que s'adona que cal saber
l'espècie abans de poder jutjar la massa. Qui digui només "la massa, perquè els
mascles són més grossos" té raó a mitges i val la pena repreguntar-li: *i un
pingüí de quatre quilos, què és?*

---

## Resum de les funcions de Pandas que han aparegut

| Funció | Què fa |
|---|---|
| `value_counts()` | compta les repeticions de cada valor d'una columna |
| `sort_index()` | ordena per l'etiqueta en comptes de per la freqüència |
| `pd.crosstab(a, b)` | taula de contingència: creua dues columnes categòriques |
| `idxmax()` | l'**índex** de la fila amb el valor màxim (`max()` dóna el valor) |
| `df.loc[i]` | selecciona una fila (o un tros de taula) per etiqueta |
| `groupby(col)` | parteix la taula en grups per aplicar-hi una operació |
| `agg(["min", "max"])` | aplica diverses operacions de cop a cada grup |
| `.plot(kind="bar")` | dibuixa directament una Series o un DataFrame |